# NumPy 章节验收测验：粮食产量分析

适用：已学基础 Python 语法和本章 NumPy 的同学。建议独立完成 60 分钟，共 100 分。只使用 Python 标准库和 NumPy；不需要 pandas、画图或网络。

**数据**：本目录 `data/粮食产量年度数据.csv`，教材原始文件的副本。单位均为**万吨**。本测验只用其中 2014—2023 年的粮食、稻谷和玉米产量。数据读取与“2023年”文字转换由下面的给定代码完成，不计分；原始文件按年份从新到旧排列。

教师专用参考答案。学生版与本文件分开保存；按顺序运行代码。答案允许其他等价 NumPy 写法。

In [ ]:
from pathlib import Path
import numpy as np

DATA = Path("data") / "粮食产量年度数据.csv"
if not DATA.is_file():
    raise FileNotFoundError("找不到数据文件：请从本测验所在文件夹启动 Jupyter，并保留 data 文件夹。")

# 教师给定的读取代码：列0年份、列1粮食、列3稻谷、列5玉米；取文件前10行。
raw = np.loadtxt(
    DATA, delimiter=",", skiprows=1, usecols=(0, 1, 3, 5),
    max_rows=10, dtype=str, encoding="utf-8-sig"
)
year_list = []
for text in raw[:, 0]:
    year_list.append(int(text.replace("年", "")))
years_raw = np.array(year_list)
grain_raw = raw[:, 1].astype(float)
rice_raw = raw[:, 2].astype(float)
corn_raw = raw[:, 3].astype(float)
print("读取记录数：", years_raw.size)
print("原始前3个年份：", years_raw[:3])

## 题1：排好年份，检查数组（20分）

1. 用 `np.argsort` 生成按年份升序排列的位置索引，并**同时**整理 `years_raw`、`grain_raw`、`rice_raw`、`corn_raw`，得到 `years`、`grain`、`rice`、`corn`。不要只排序某一列。
2. 输出 `years` 和 `grain` 的 `shape`，以及整理后的首年、末年及其粮食产量。用一句话说明为什么四个数组必须使用同一组位置索引。

In [ ]:
order = np.argsort(years_raw)
years = years_raw[order]
grain = grain_raw[order]
rice = rice_raw[order]
corn = corn_raw[order]
print("年份数组：", years)
print("years.shape：", years.shape, "grain.shape：", grain.shape)
print("首年及粮食产量：", years[0], grain[0], "万吨")
print("末年及粮食产量：", years[-1], grain[-1], "万吨")

**参考解释：** 四个数组的相同位置表示同一年；若只重排年份，产量会与错误年份配对。

## 题2：统计粮食产量（25分）

1. 计算 2014—2023 年粮食产量的平均值，保留两位小数输出，写明单位。
2. 找出粮食产量最高的年份及其产量。提示：`np.argmax` 返回的是**位置**。
3. 使用布尔条件找出粮食产量**严格高于**这十年均值的年份，输出年份数组和年份个数。

In [ ]:
avg_grain = np.mean(grain)
max_pos = np.argmax(grain)
above = years[grain > avg_grain]
print("平均年粮食产量：", round(avg_grain, 2), "万吨")
print("最高年份和产量：", years[max_pos], grain[max_pos], "万吨")
print("高于均值的年份：", above, "共", above.size, "年")

## 题3：比较稻谷和玉米（25分）

1. 用数组运算计算各年的“玉米产量－稻谷产量”（单位：万吨），找出差值最大的年份和差值。
2. 分别计算 2014 年、2023 年**玉米产量占同年粮食总产量的百分比**，保留两位小数。
3. 用一句话说明该占比是上升还是下降，并写出分子和分母分别是什么。注意不要把不同年份的数据配在一起。

In [ ]:
gap = corn - rice
gap_pos = np.argmax(gap)
share_2014 = corn[0] / grain[0] * 100
share_2023 = corn[-1] / grain[-1] * 100
print("最大差值年份和差值：", years[gap_pos], round(gap[gap_pos], 2), "万吨")
print("2014年玉米占比：", round(share_2014, 2), "%")
print("2023年玉米占比：", round(share_2023, 2), "%")

**参考解释：** 2014 年约 39.05%，2023 年约 41.54%，占比上升。分子是当年玉米产量，分母是同年粮食总产量；这是两个同单位量的比例。

## 题4：分析相邻年份的变化（30分）

1. 用切片计算粮食产量的逐年增长量：当年产量减上年产量。把每项增长量与**当年年份**对应，输出年份数组与增长量数组。2014—2023 年有多少项增长量？
2. 找出产量下降的年份及对应增长量；找出增长量最大的年份和增长量。
3. 判断“2014—2023 年粮食产量每年都上升”是否正确。用本题结果写一句有年份、有单位的解释。

In [ ]:
growth_years = years[1:]
growth = grain[1:] - grain[:-1]
decline_mask = growth < 0
biggest_pos = np.argmax(growth)
print("增长量所属年份：", growth_years)
print("逐年增长量（万吨）：", np.round(growth, 2))
print("增长量项数：", growth.size)
print("下降年份：", growth_years[decline_mask])
print("对应增长量（万吨）：", np.round(growth[decline_mask], 2))
print("增长量最大年份和增长量：", growth_years[biggest_pos], round(growth[biggest_pos], 2), "万吨")

**参考解释：** 说法不正确。2016 年粮食产量比 2015 年少 16.76 万吨，2018 年比 2017 年少 371.51 万吨。2014—2023 年虽然期末高于期初，但不能说每年都上升。